# Chapter 7 · How do you know it works?

A broken model doesn't crash. It trains, prints a falling loss, and writes something. In
chapter 4, a broken model even scored *better* than a correct one. So how do you know your
code is right before you spend ten minutes of GPU time on it?

With **tests**: small functions that each check one thing that must be true of correct
code, and that run in seconds. In this chapter you plant real bugs in tinyGPT and find
out which tests catch them, and which bug gets past all of them.

| | |
|---|---|
| **Open box** | the model's tests: `tests/test_model.py` |
| **The number** | 4.17 again: an untrained model that doesn't start near it is broken |
| **Time** | about an hour |
| **Python you need** | functions and `assert` |
| **GPU** | not needed |

**How this chapter works.** Five steps, in order:

1. **Predict** `everyone`: guess before you run anything. A wrong guess is useful: it shows you what you expected.
2. **Run** `everyone`: run the cells and compare with your guesses.
3. **Investigate** `everyone`: open the box. Read the code that just ran and answer a few questions.
4. **Modify** `most people`: change one thing, predict what will happen, then explain what did.
5. **Make** `optional challenge`: build this chapter's piece yourself. A check tells you when it's right.

Run a cell with **Shift+Enter**. If the notebook restarts, run the setup cell again first.

## Setup

In [ ]:
#@title Setup: run this cell first (about 30 seconds) { display-mode: "form" }
# Downloads tinyGPT (on Colab) and installs it. You don't need to read or change this cell.
import importlib, os, subprocess, sys
from pathlib import Path

def run(*command):
    done = subprocess.run(command, capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(f"{' '.join(command)} failed:\n{done.stdout}{done.stderr}")

if "google.colab" in sys.modules:
    repo = Path("/content/tinyGPT")
    if not repo.exists():
        run("git", "clone", "-q", "https://github.com/cgpan/tinyGPT.git", str(repo))
    root = repo / "entry"
else:
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "src" / "tinygpt").is_dir()), None)
    if root is None:
        raise FileNotFoundError("Can't find the tinyGPT folder. Open this notebook from inside it, or use Colab.")
os.chdir(root)
run(sys.executable, "-m", "pip", "install", "-q", ".")
importlib.invalidate_caches()

import tinygpt
tinygpt.print_setup()

## 1 · Predict `everyone`

tinyGPT comes with about a hundred tests, in the `tests/` folder. Here are five bugs.
Each one is a mistake people really make when they write a transformer.

In [ ]:
for name, description in tinygpt.bugs.BUGS.items():
    print(f"{name:<18} {description}")

For each bug: will at least one test catch it?

In [ ]:
guess_no_mask = None              # True if you think a test will catch it, False if not
guess_softmax_wrong_way = None
guess_no_shift = None
guess_big_init = None
guess_no_position = None

## 2 · Run `everyone`

First the tests on tinyGPT as it is. A dot is a test that passed.

In [ ]:
tinygpt.run_tests()

Now plant each bug in turn and run the tests again. Each `FAILED` line names a test that
caught the bug. The five runs take about a minute.

In [ ]:
for bug in tinygpt.bugs.BUGS:
    tinygpt.run_tests(bug=bug)
    print()

Which of your guesses were right? Which bug got past every test?

*Your explanation (double-click to edit):*

## 3 · Investigate `everyone`

### Open the box

The tests for the model. Each one checks a property that every correct model has,
trained or not.

In [ ]:
tinygpt.show("tests/test_model.py")

1. `test_untrained_gpt_loss_is_close_to_uniform_guessing`: why should an untrained model
   score about ln 65 = 4.17? Which bug did this test catch? The next cell shows how far
   off that bug makes it.
2. `test_attention_cannot_see_the_future` changes only the *last* character. What must
   stay exactly the same? It caught two different bugs. How can a softmax that runs the
   wrong way let the future leak in?
3. `test_pos_emb_switch_removes_the_position_table` *sounds* as if it tests positions.
   What does it actually check? Why didn't it catch `no_position`?
4. None of these tests trains a model and checks for 1.49. Why test small properties of
   an untrained model instead?

In [ ]:
import torch

tok, train_ids, val_ids = tinygpt.load_data()
cfg = tinygpt.load_config("configs/small.yaml")
x, y = tinygpt.get_batch(val_ids, cfg.block_size, 32)

torch.manual_seed(1337)
correct = tinygpt.GPT(cfg).eval()
with tinygpt.bugs.planted("big_init"):             # inside this block, tinyGPT has the bug
    torch.manual_seed(1337)
    broken = tinygpt.GPT(cfg).eval()

with torch.no_grad():
    print(f"untrained, correct:      loss {correct(x, y)[1].item():.2f}")
    print(f"untrained, big_init bug: loss {broken(x, y)[1].item():.2f}")

*Your explanation (double-click to edit):*

## 4 · Modify `most people`

**A. What a bug does to training.** Train the one-block model from chapter 4 twice:
once as it is, and once with the `no_shift` bug planted. **Predict** the bugged model's
loss curve, and what it will write. Each run takes about a minute.

In [ ]:
one_block = tinygpt.load_config("configs/small.yaml", n_layer=1)
_, correct_history = tinygpt.train(one_block)
with tinygpt.bugs.planted("no_shift"):
    shifty, bug_history = tinygpt.train(one_block)
tinygpt.plot_history({"correct": correct_history, "no_shift bug": bug_history})

In [ ]:
print(repr(tinygpt.sample(shifty, tok, prompt="ROMEO:", length=100, seed=0)))

A loss of almost exactly zero, and it can't write. With `no_shift`, the answer at each
position is the character the model is already reading, so it learns to copy. *Too good
to be true* is the most common sign of a bug, and a one-second test caught this one
before any training.

*Your explanation (double-click to edit):*

**B. A test written by an AI.** Ask an AI assistant such as ChatGPT or Claude:

> *Write a short Python test function, with no arguments and no pytest fixtures, that
> checks that `tinygpt.get_batch(ids, block_size, batch_size)` returns targets `y` that
> are the inputs `x` moved along by one position. `ids` is a 1-D torch tensor of integers.*

Paste its function into the cell below, in place of the example, and change the name in
the last line to match. `check.catches` runs it twice: on tinyGPT as it is, where it
should pass, and with the `no_shift` bug planted, where it should fail.

In [ ]:
def test_from_ai():                    # an example of a weak test: replace it with the AI's
    x, y = tinygpt.get_batch(torch.arange(100), block_size=8, batch_size=4)
    assert x.shape == y.shape          # only checks the shapes

tinygpt.check.catches(test_from_ai, bug="no_shift")

Did the AI's test catch the bug? Was it testing the right thing, or only that the code runs?

*Your explanation (double-click to edit):*

## 5 · Make `optional challenge`

Write the test that catches `no_position`, the bug that got past every test. A hint:
with the position table, the same character at two different positions starts out as
two different vectors, so the model gives them different scores. Without it, a row of
one repeated character looks the same at every position.

In [ ]:
def test_position_matters():
    """The same character at different positions should get different scores."""
    torch.manual_seed(0)
    cfg = tinygpt.Config(vocab_size=65, block_size=8, n_layer=1, n_head=2, n_embd=32, dropout=0.0)
    model = tinygpt.GPT(cfg).eval()
    # your code here: give the model one character repeated 8 times (torch.full makes that),
    # and assert that its scores at the first and the last position are not the same.
    # Compare with torch.allclose(..., atol=1e-3): tiny rounding differences don't count
    assert True                       # replace this line

tinygpt.check.catches(test_position_matters, bug="no_position")

Going further: add your test to `tests/test_model.py` (on Colab it is in the Files panel,
under `tinyGPT/tests`) and run `tinygpt.run_tests(bug="no_position")` again. Now the suite
catches every bug.

**Stuck?** Try first, then compare with [the solution](https://github.com/cgpan/tinyGPT/blob/main/entry/solutions/07_testing.md) (also in the `solutions/` folder of your copy).

## Where the number stands

In [ ]:
tinygpt.scoreboard({"one block, correct": correct_history[-1]["val_loss"],
                    "one block, no_shift bug": bug_history[-1]["val_loss"]})

The bugged model is far to the right of tinyGPT. When a number is this much better than
expected, look for the bug before you celebrate.

## Common questions

**What is pytest?** The standard Python tool for running tests. It finds every function
whose name starts with `test_`, runs it, and reports the ones that fail.

**How is a test different from the loss?** A test checks that the code does what you
meant: pass or fail, in seconds. The loss measures how good a model is. You need both,
and the `no_shift` run shows why: a great loss from broken code.

**Should every bug get a test?** Every bug you find, yes: write the test that would have
caught it, so it can never come back unnoticed.

**Can an AI write my tests?** It can draft them quickly. Then do what `check.catches`
does: plant the bug, and make sure the test fails. A test that never fails checks
nothing.

**How are the bugs planted?** `tinygpt/bugs.py` swaps one function for a broken copy,
only inside a `with tinygpt.bugs.planted(...)` block or a test run, and swaps it back
afterwards.

## If something breaks

| What you see | What to do |
|---|---|
| `FileNotFoundError: No tests/ folder here` | Run the setup cell: it moves into the tinyGPT folder. |
| "fails on tinyGPT as it is, so it would blame correct code" | Your test fails even without the bug. Read the message after the colon. |
| The AI's test takes an argument such as `tmp_path` | Ask it for a function with no arguments and no pytest fixtures. |
| `NameError` | Run the cells above it, in order. In Colab: Runtime > Run before. |

## Next

**Chapter 8 · Meet a real one** (optional). Put tinyGPT next to GPT-2, a real model
with 124 million parameters, and compare them fairly.